In [1]:
import numpy as np
import qutip as qt
from quspin.basis import spin_basis_1d
from quspin.operators import hamiltonian

import geodesiq as gq

In [67]:
# ----- Define ControlModel -----
def ising_model(lam, L, hx, hz):
    """
    Constructs the Ising ControlModel with transverse (hx) and longitudinal (hz) fields.
    """
    zz_list = [[lam, i, i + 1] for i in range(L - 1)]
    z_list = [[lam * hz, i] for i in range(L)]
    x_list = [[(1 - lam) * hx, i] for i in range(L)]

    static = [["zz", zz_list], ["z", z_list], ["x", x_list]]

    basis = spin_basis_1d(L, pblock=1)
    H = hamiltonian(static, [], basis=basis, dtype=np.float64, check_symm=False, check_herm=False)

    return H.toarray()

n_rep = 5
n_ts = 2 ** 5 + 1  # time slices
amp_lbound, amp_ubound = 0.1, 0.9
L, hx, hz = 4, 1, .8
tf = 250

alpha = 2
beta = 2

fid_err_targ = 0.00031

index_0 = 6
index_t = 6

H_d = qt.Qobj(ising_model(0, L, hx, hz))
H_c = qt.Qobj(ising_model(1, L, hx, hz)) - H_d

In [6]:
H_initial = qt.Qobj(ising_model(amp_lbound, L, hx, hz))
H_final = qt.Qobj(ising_model(amp_ubound, L, hx, hz))
psi0 = H_initial.eigenstates()[1][index_0]
psit = H_final.eigenstates()[1][index_t]
model = gq.ControlModel(ising_model)
model.set_parameters(L=L, hx=hx, hz=hz)
model.set_control(control_name='lam', pulse_initial=amp_lbound, pulse_final=amp_ubound,
                  initial_state=index_0, alpha=alpha, beta=beta, num_steps=n_ts)
model.solve_problem(pulse_accuracy=1000)
dynamics = gq.Dynamics(tf, model)

In [47]:
times = np.linspace(0.0, tf, 1001)
decomposition = gq.decompose_hamiltonian(dynamics._get_ham, times, drift="mean", rtol=1e-10)

print(f"Rank: {decomposition.rank}")
print(f"Relative residual error: {decomposition.relative_residual_error:.3e}")
print(f"Relative total error: {decomposition.relative_total_error:.3e}")

Rank: 1
Relative residual error: 2.132e-15
Relative total error: 8.002e-16


In [48]:
errors = []

for i, t in enumerate(times):
    H_exact = dynamics._get_ham(float(t))
    H_approx = decomposition.reconstruct_sample(i)

    errors.append((H_exact - H_approx).norm())

print(f"Maximum error: {np.max(errors):.3e}")
print(f"Mean error:    {np.mean(errors):.3e}")

Maximum error: 2.230e-14
Mean error:    1.438e-14


In [51]:
qt.Qobj(ising_model(amp_lbound, L, hx, hz))

Quantum object: dims=[[10], [10]], shape=(10, 10), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.62        1.27279221  1.27279221  0.          0.          0.
   0.          0.          0.          0.        ]
 [ 1.27279221  0.26        0.          0.9         0.9         0.
   0.          1.27279221  0.          0.        ]
 [ 1.27279221  0.          0.06        0.9         0.9         1.27279221
   0.          0.          0.          0.        ]
 [ 0.          0.9         0.9         0.1         0.          0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.9         0.9         0.         -0.3         0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.          1.27279221  0.          0.         -0.1
   1.27279221  0.          0.          0.        ]
 [ 0.          0.          0.          0.9         0.9         1.27279221
  -0.06        0.          0.          1.27279221]
 [ 0.          1.27279221  0.          0.          0.          

In [49]:
dynamics._control_sol[0]

np.float64(0.1)

In [50]:
dynamics._get_ham(float(0))

Quantum object: dims=[[10], [10]], shape=(10, 10), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.62        1.27279221  1.27279221  0.          0.          0.
   0.          0.          0.          0.        ]
 [ 1.27279221  0.26        0.          0.9         0.9         0.
   0.          1.27279221  0.          0.        ]
 [ 1.27279221  0.          0.06        0.9         0.9         1.27279221
   0.          0.          0.          0.        ]
 [ 0.          0.9         0.9         0.1         0.          0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.9         0.9         0.         -0.3         0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.          1.27279221  0.          0.         -0.1
   1.27279221  0.          0.          0.        ]
 [ 0.          0.          0.          0.9         0.9         1.27279221
  -0.06        0.          0.          1.27279221]
 [ 0.          1.27279221  0.          0.          0.          

In [56]:
qt.Qobj(ising_model(amp_lbound, L, hx, hz))

Quantum object: dims=[[10], [10]], shape=(10, 10), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.62        1.27279221  1.27279221  0.          0.          0.
   0.          0.          0.          0.        ]
 [ 1.27279221  0.26        0.          0.9         0.9         0.
   0.          1.27279221  0.          0.        ]
 [ 1.27279221  0.          0.06        0.9         0.9         1.27279221
   0.          0.          0.          0.        ]
 [ 0.          0.9         0.9         0.1         0.          0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.9         0.9         0.         -0.3         0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.          1.27279221  0.          0.         -0.1
   1.27279221  0.          0.          0.        ]
 [ 0.          0.          0.          0.9         0.9         1.27279221
  -0.06        0.          0.          1.27279221]
 [ 0.          1.27279221  0.          0.          0.          

In [70]:
qt.Qobj(H_d + H_c * amp_lbound)

Quantum object: dims=[[10], [10]], shape=(10, 10), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.62        1.27279221  1.27279221  0.          0.          0.
   0.          0.          0.          0.        ]
 [ 1.27279221  0.26        0.          0.9         0.9         0.
   0.          1.27279221  0.          0.        ]
 [ 1.27279221  0.          0.06        0.9         0.9         1.27279221
   0.          0.          0.          0.        ]
 [ 0.          0.9         0.9         0.1         0.          0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.9         0.9         0.         -0.3         0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.          1.27279221  0.          0.         -0.1
   1.27279221  0.          0.          0.        ]
 [ 0.          0.          0.          0.9         0.9         1.27279221
  -0.06        0.          0.          1.27279221]
 [ 0.          1.27279221  0.          0.          0.          

In [43]:
coefficients[0, 0]

np.float64(4.8752142334610165)

In [42]:
H_drift = decomposition.H_d
H_controls = decomposition.H_controls
coefficients = decomposition.coefficients

H_drift + H_controls[0] * coefficients[0, 0]

Quantum object: dims=[[10], [10]], shape=(10, 10), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.62        1.27279221  1.27279221  0.          0.          0.
   0.          0.          0.          0.        ]
 [ 1.27279221  0.26        0.          0.9         0.9         0.
   0.          1.27279221  0.          0.        ]
 [ 1.27279221  0.          0.06        0.9         0.9         1.27279221
   0.          0.          0.          0.        ]
 [ 0.          0.9         0.9         0.1         0.          0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.9         0.9         0.         -0.3         0.
   0.9         0.          0.9         0.        ]
 [ 0.          0.          1.27279221  0.          0.         -0.1
   1.27279221  0.          0.          0.        ]
 [ 0.          0.          0.          0.9         0.9         1.27279221
  -0.06        0.          0.          1.27279221]
 [ 0.          1.27279221  0.          0.          0.          

In [44]:
relative_singular_values = decomposition.singular_values / decomposition.singular_values[0]

print(relative_singular_values)

[1.00000000e+00 1.87815791e-15 2.19435940e-16 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.99200722e-17 9.99200722e-17 9.99200722e-17
 9.99200722e-17 9.992007